In [ ]:
!nvcc --version
!nvidia-smi

In [ ]:
!pip install -U transformers accelerate bitsandbytes pandas matplotlib

##Imports + Config

In [ ]:
import os
import gc
import csv
import time
import threading
from dataclasses import dataclass, asdict
from typing import List, Dict, Any

import torch
import pandas as pd
import matplotlib.pyplot as plt

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TextIteratorStreamer,
    BitsAndBytesConfig,
)

In [ ]:
# ============================================================
# Colab T4 Config
# ============================================================

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

RESULT_DIR = "/content/day24_quant_results"
os.makedirs(RESULT_DIR, exist_ok=True)

CSV_PATH = os.path.join(RESULT_DIR, "benchmark_results.csv")

PROMPTS = [
    "Explain what quantization means for LLM inference in simple terms.",
    "Why can quantization reduce GPU memory usage?",
    "Compare FP16 inference and INT8 inference for large language models.",
    "Write a short paragraph about the trade-off between latency and output quality.",
]

MAX_NEW_TOKENS_LIST = [32, 64, 128]

NUM_WARMUP = 1
NUM_RUNS = 3

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## Result Dataclass

In [ ]:
@dataclass
class BenchmarkResult:
    mode: str
    model_name: str
    quantization: str
    prompt_id: int
    prompt_chars: int
    input_tokens: int
    output_tokens: int
    max_new_tokens: int

    ttft_ms: float
    total_latency_ms: float
    tpot_ms: float
    throughput_tok_s: float

    peak_allocated_gb: float
    peak_reserved_gb: float

    output_text: str
    quality_note: str

## Utility Functions

In [ ]:
def clear_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def get_peak_memory_gb():
    if not torch.cuda.is_available():
        return 0.0, 0.0

    allocated = torch.cuda.max_memory_allocated() / 1024**3
    reserved = torch.cuda.max_memory_reserved() / 1024**3
    return allocated, reserved


def count_tokens(tokenizer, text: str) -> int:
    if len(text.strip()) == 0:
        return 0
    return len(tokenizer(text, return_tensors="pt")["input_ids"][0])


def save_results_to_csv(results: List[BenchmarkResult], path: str):
    if not results:
        return

    fieldnames = list(asdict(results[0]).keys())

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in results:
            writer.writerow(asdict(r))

    print(f"Saved results to: {path}")

## Load Model(Todo)

In [ ]:
def load_model_and_tokenizer(
    model_name: str,
    quantization: str,
):
    """
    quantization:
    - "fp16"
    - "int8"

    TODO:
    你需要补全 tokenizer 和 model loading。
    """

    print("=" * 80)
    print(f"Loading model: {model_name}")
    print(f"Quantization mode: {quantization}")
    print("=" * 80)

    clear_gpu_memory()

    # ========================================================
    # TODO 1: Load tokenizer
    # ========================================================
    tokenizer = None

    # Hint:
    # tokenizer = AutoTokenizer.from_pretrained(
    #     model_name,
    #     trust_remote_code=True,
    # )
    #
    # if tokenizer.pad_token is None:
    #     tokenizer.pad_token = tokenizer.eos_token


    # ========================================================
    # TODO 2: Load model
    # ========================================================
    model = None

    if quantization == "fp16":
        # ----------------------------------------------------
        # TODO:
        # Load FP16 baseline model
        # ----------------------------------------------------
        #
        # Hint:
        # model = AutoModelForCausalLM.from_pretrained(
        #     model_name,
        #     torch_dtype=torch.float16,
        #     device_map="auto",
        #     trust_remote_code=True,
        # )
        pass

    elif quantization == "int8":
        # ----------------------------------------------------
        # TODO:
        # Load INT8 quantized model with bitsandbytes
        # ----------------------------------------------------
        #
        # Hint:
        # bnb_config = BitsAndBytesConfig(
        #     load_in_8bit=True,
        # )
        #
        # model = AutoModelForCausalLM.from_pretrained(
        #     model_name,
        #     quantization_config=bnb_config,
        #     device_map="auto",
        #     trust_remote_code=True,
        # )
        pass

    else:
        raise ValueError(f"Unknown quantization mode: {quantization}")

    # ========================================================
    # TODO 3: eval mode
    # ========================================================
    # model.eval()

    print("Model loaded.")
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"Current allocated memory: {allocated:.3f} GB")
        print(f"Current reserved memory:   {reserved:.3f} GB")

    return tokenizer, model

## Prompt Formatting

In [ ]:
def build_prompt(tokenizer, raw_prompt: str) -> str:
    """
    TODO:
    construct prompt using Qwen's chat template.

    Return:
        formatted prompt string
    """

    # ========================================================
    # TODO:
    # apply tokenizer.apply_chat_template
    # ========================================================

    # Hint:
    # messages = [
    #     {"role": "system", "content": "You are a helpful assistant."},
    #     {"role": "user", "content": raw_prompt},
    # ]
    #
    # prompt = tokenizer.apply_chat_template(
    #     messages,
    #     tokenize=False,
    #     add_generation_prompt=True,
    # )
    #
    # return prompt

    return raw_prompt

## single inference + TTFT/TPOT measure

In [ ]:
@torch.no_grad()
def run_single_inference_streaming(
    tokenizer,
    model,
    raw_prompt: str,
    max_new_tokens: int,
) -> Dict[str, Any]:
    """
    Run one generation and measure:

    - TTFT
    - total latency
    - TPOT
    - throughput
    - peak GPU memory

    用 TextIteratorStreamer 来估算 TTFT。
    """

    clear_gpu_memory()

    formatted_prompt = build_prompt(tokenizer, raw_prompt)

    # ========================================================
    # TODO 1: Tokenize input
    # ========================================================
    inputs = None
    input_tokens = 0

    # Hint:
    # inputs = tokenizer(
    #     formatted_prompt,
    #     return_tensors="pt",
    # )
    #
    # inputs = {k: v.to(model.device) for k, v in inputs.items()}
    # input_tokens = inputs["input_ids"].shape[-1]


    # ========================================================
    # TODO 2: Create streamer
    # ========================================================
    streamer = None

    # Hint:
    # streamer = TextIteratorStreamer(
    #     tokenizer,
    #     skip_prompt=True,
    #     skip_special_tokens=True,
    # )


    # ========================================================
    # TODO 3: Generation kwargs
    # ========================================================
    generation_kwargs = None

    # Hint:
    # generation_kwargs = dict(
    #     **inputs,
    #     streamer=streamer,
    #     max_new_tokens=max_new_tokens,
    #     do_sample=False,
    #     pad_token_id=tokenizer.eos_token_id,
    # )


    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start_time = time.perf_counter()
    first_token_time = None
    output_chunks = []

    # ========================================================
    # TODO 4: Start generation in another thread
    # ========================================================
    thread = None

    # Hint:
    # thread = threading.Thread(
    #     target=model.generate,
    #     kwargs=generation_kwargs,
    # )
    # thread.start()


    # ========================================================
    # TODO 5: Consume streamer and capture TTFT
    # ========================================================
    #
    # Hint:
    # for new_text in streamer:
    #     if first_token_time is None:
    #         if torch.cuda.is_available():
    #             torch.cuda.synchronize()
    #         first_token_time = time.perf_counter()
    #
    #     output_chunks.append(new_text)


    # ========================================================
    # TODO 6: Wait for generation thread
    # ========================================================
    # thread.join()


    if torch.cuda.is_available():
        torch.cuda.synchronize()

    end_time = time.perf_counter()

    output_text = "".join(output_chunks)

    if first_token_time is None:
        first_token_time = end_time

    total_latency_s = end_time - start_time
    ttft_s = first_token_time - start_time

    # ========================================================
    # TODO 7: Count output tokens
    # ========================================================
    output_tokens = 0

    # Hint:
    # output_tokens = count_tokens(tokenizer, output_text)


    # TPOT: exclude first token time
    if output_tokens > 1:
        tpot_s = (total_latency_s - ttft_s) / (output_tokens - 1)
    else:
        tpot_s = 0.0

    throughput = output_tokens / total_latency_s if total_latency_s > 0 else 0.0

    peak_allocated_gb, peak_reserved_gb = get_peak_memory_gb()

    return {
        "formatted_prompt": formatted_prompt,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "output_text": output_text,
        "ttft_ms": ttft_s * 1000,
        "total_latency_ms": total_latency_s * 1000,
        "tpot_ms": tpot_s * 1000,
        "throughput_tok_s": throughput,
        "peak_allocated_gb": peak_allocated_gb,
        "peak_reserved_gb": peak_reserved_gb,
    }

## Subjective Quality Note

In [ ]:
def subjective_quality_note(output_text: str) -> str:
    if len(output_text.strip()) == 0:
        return "empty output / failed"

    if len(output_text.strip()) < 30:
        return "too short / possibly degraded"

    return "manual review needed"

## Benchmark Loop

In [ ]:
def benchmark_one_mode(
    model_name: str,
    quantization: str,
    mode_name: str,
) -> List[BenchmarkResult]:

    tokenizer, model = load_model_and_tokenizer(
        model_name=model_name,
        quantization=quantization,
    )

    all_results = []

    # Warmup
    print("\nRunning warmup...")
    for _ in range(NUM_WARMUP):
        _ = run_single_inference_streaming(
            tokenizer=tokenizer,
            model=model,
            raw_prompt=PROMPTS[0],
            max_new_tokens=32,
        )

    print("\nRunning benchmark...")

    for max_new_tokens in MAX_NEW_TOKENS_LIST:
        for prompt_id, prompt in enumerate(PROMPTS):

            repeated_metrics = []

            for run_id in range(NUM_RUNS):
                print(
                    f"[{mode_name}] "
                    f"max_new_tokens={max_new_tokens}, "
                    f"prompt_id={prompt_id}, "
                    f"run={run_id}"
                )

                metrics = run_single_inference_streaming(
                    tokenizer=tokenizer,
                    model=model,
                    raw_prompt=prompt,
                    max_new_tokens=max_new_tokens,
                )

                repeated_metrics.append(metrics)

            numeric_keys = [
                "input_tokens",
                "output_tokens",
                "ttft_ms",
                "total_latency_ms",
                "tpot_ms",
                "throughput_tok_s",
                "peak_allocated_gb",
                "peak_reserved_gb",
            ]

            avg = {}
            for key in numeric_keys:
                avg[key] = sum(m[key] for m in repeated_metrics) / len(repeated_metrics)

            output_text = repeated_metrics[-1]["output_text"]

            result = BenchmarkResult(
                mode=mode_name,
                model_name=model_name,
                quantization=quantization,
                prompt_id=prompt_id,
                prompt_chars=len(prompt),
                input_tokens=int(avg["input_tokens"]),
                output_tokens=int(avg["output_tokens"]),
                max_new_tokens=max_new_tokens,

                ttft_ms=avg["ttft_ms"],
                total_latency_ms=avg["total_latency_ms"],
                tpot_ms=avg["tpot_ms"],
                throughput_tok_s=avg["throughput_tok_s"],

                peak_allocated_gb=avg["peak_allocated_gb"],
                peak_reserved_gb=avg["peak_reserved_gb"],

                output_text=output_text,
                quality_note=subjective_quality_note(output_text),
            )

            all_results.append(result)

    del model
    del tokenizer
    clear_gpu_memory()

    return all_results

## plotting

In [ ]:
def plot_metric_bar(
    df: pd.DataFrame,
    metric: str,
    ylabel: str,
    filename: str,
):
    grouped = (
        df.groupby(["mode", "max_new_tokens"])[metric]
        .mean()
        .reset_index()
    )

    pivot = grouped.pivot(
        index="max_new_tokens",
        columns="mode",
        values=metric,
    )

    ax = pivot.plot(kind="bar", figsize=(8, 5))

    ax.set_title(metric)
    ax.set_xlabel("max_new_tokens")
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, filename), dpi=200)
    plt.show()


def plot_metric_line(
    df: pd.DataFrame,
    metric: str,
    ylabel: str,
    filename: str,
):
    grouped = (
        df.groupby(["mode", "max_new_tokens"])[metric]
        .mean()
        .reset_index()
    )

    plt.figure(figsize=(8, 5))

    for mode in grouped["mode"].unique():
        sub = grouped[grouped["mode"] == mode]
        plt.plot(
            sub["max_new_tokens"],
            sub[metric],
            marker="o",
            label=mode,
        )

    plt.title(metric)
    plt.xlabel("max_new_tokens")
    plt.ylabel(ylabel)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, filename), dpi=200)
    plt.show()


def make_all_plots(csv_path: str):
    df = pd.read_csv(csv_path)

    plot_metric_bar(
        df,
        metric="ttft_ms",
        ylabel="TTFT ms",
        filename="ttft_bar.png",
    )

    plot_metric_bar(
        df,
        metric="tpot_ms",
        ylabel="TPOT ms/token",
        filename="tpot_bar.png",
    )

    plot_metric_bar(
        df,
        metric="throughput_tok_s",
        ylabel="Output tokens / second",
        filename="throughput_bar.png",
    )

    plot_metric_bar(
        df,
        metric="peak_allocated_gb",
        ylabel="Peak allocated GPU memory GB",
        filename="peak_allocated_memory_bar.png",
    )

    plot_metric_bar(
        df,
        metric="peak_reserved_gb",
        ylabel="Peak reserved GPU memory GB",
        filename="peak_reserved_memory_bar.png",
    )

    plot_metric_line(
        df,
        metric="total_latency_ms",
        ylabel="Total latency ms",
        filename="total_latency_line.png",
    )

    print(f"\nPlots saved to: {RESULT_DIR}")

## Run Benchmark(baseline_fp16 + quantized_int8)

In [ ]:
all_results = []

# ============================================================
# 1. FP16 baseline
# ============================================================

baseline_results = benchmark_one_mode(
    model_name=MODEL_NAME,
    quantization="fp16",
    mode_name="baseline_fp16",
)

all_results.extend(baseline_results)
save_results_to_csv(all_results, CSV_PATH)


# ============================================================
# 2. INT8 quantized
# ============================================================

int8_results = benchmark_one_mode(
    model_name=MODEL_NAME,
    quantization="int8",
    mode_name="quantized_int8",
)

all_results.extend(int8_results)
save_results_to_csv(all_results, CSV_PATH)


# ============================================================
# 3. Plot
# ============================================================

make_all_plots(CSV_PATH)

## Summary Table

In [ ]:
df = pd.read_csv(CSV_PATH)

summary = (
    df.groupby("mode")[
        [
            "ttft_ms",
            "total_latency_ms",
            "tpot_ms",
            "throughput_tok_s",
            "peak_allocated_gb",
            "peak_reserved_gb",
        ]
    ]
    .mean()
    .reset_index()
)

summary

## Show Detailed Results

In [ ]:
df[
    [
        "mode",
        "quantization",
        "prompt_id",
        "max_new_tokens",
        "input_tokens",
        "output_tokens",
        "ttft_ms",
        "tpot_ms",
        "throughput_tok_s",
        "peak_allocated_gb",
        "quality_note",
    ]
]

In [ ]:
for i, row in df.iterrows():
    print("=" * 100)
    print("Mode:", row["mode"])
    print("Prompt ID:", row["prompt_id"])
    print("Max new tokens:", row["max_new_tokens"])
    print("Output:")
    print(row["output_text"])
    print()